<a href="https://colab.research.google.com/github/mailan48692004-web/Climate-Risk-Monitoring-and-Site-Prioritisation/blob/main/notebook/notebook_1_data_preparation_and_index_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESG Risk Analytics: Climate Risk Monitoring and Early Warning for Business Operations in Australia

## Business Context
Businesses operating across multiple Australian locations face different levels of climate-related disruption risk. These risks are shaped not only by external hazard exposure, but also by internal operational vulnerability, resilience, and the business importance of each site. This project uses a synthetic site-month dataset to assess how climate and operational factors combine to influence site-level disruption risk and to support more informed monitoring and decision-making.

## Project Aim
The aim of this project is to analyse site-level climate-related operational risk, develop practical monitoring indicators, and create an early-warning framework that helps identify higher-risk sites and supports risk mitigation planning.

## Objectives
1. Clean and standardise the site-month dataset to create a reliable foundation for analysis.  
2. Examine climate exposure, operational vulnerability, resilience, and business importance across sites.  
3. Construct composite indicators, including a climate risk index, vulnerability score, business impact score, and overall risk score.  
4. Explore how these factors relate to next-month disruption risk.  
5. Develop a monitoring and early-warning framework to support site prioritisation and business decision-making.

## Dataset Overview
The dataset is structured at the site-month level, where each row represents one business site observed in one month. It includes site profile variables, operational characteristics, climate and hazard indicators, historical disruption measures, and a lead-1 disruption outcome variable for the following month. This structure supports both cross-sectional comparison and time-based risk monitoring.

## Methodological Note
This project is based on a synthetic dataset, and the indicators used are assumption-based rather than empirically calibrated from real company records. The analysis is designed for relative risk ranking, monitoring, and business interpretation rather than causal inference or real-world forecasting validation. Composite indices are used to summarise different dimensions of site-level risk in a transparent and practical way.

## Expected Outputs
The final outputs of this project include a cleaned and processed analysis-ready dataset, composite risk indicators, exploratory analysis of site-level climate-related risk patterns, a dashboard for monitoring and prioritisation, and a practical early-warning and mitigation framework for business decision support.

## Scope of This Notebook

This notebook focuses on the data preparation stage of the project.  
It covers data cleaning, standardisation, validation, feature engineering, and the construction of composite indices used for later risk analysis and monitoring.

Exploratory analysis, risk segmentation, visualisation, and early-warning interpretation are developed in the following notebook.

In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/ESG Project/Data/climate_operational_risk_dataset.xlsx"
df = pd.read_excel(file_path)
df.head()
df.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(6480, 20)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6480 entries, 0 to 6479
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   month                       6480 non-null   datetime64[ns]
 1   site_id                     6480 non-null   object        
 2   state                       6480 non-null   object        
 3   region                      6480 non-null   object        
 4   site_type                   6480 non-null   object        
 5   staff_count                 6480 non-null   int64         
 6   monthly_volume              6480 non-null   int64         
 7   criticality_score           6480 non-null   int64         
 8   supplier_dependency_score   6480 non-null   float64       
 9   backup_capacity             6480 non-null   float64       
 10  emergency_plan_maturity     6480 non-null   int64         
 11  building_age_years          6480 non-null   int64       

In [ ]:
df.isna().sum().sort_values(ascending=False)

,0
disruption_flag_next_month,180
month,0
state,0
site_id,0
region,0
site_type,0
monthly_volume,0
staff_count,0
supplier_dependency_score,0
backup_capacity,0


In [ ]:
df.duplicated(subset=["month", "site_id"]).sum()

np.int64(0)

In [ ]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
month,6480,NaN,NaN,NaN,2024-06-16 02:39:59.999999744,2023-01-01 00:00:00,2023-09-23 12:00:00,2024-06-16 00:00:00,2025-03-08 18:00:00,2025-12-01 00:00:00,NaN
site_id,6480,180,S0001,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,6480,7,NSW,2232,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,6480,22,Riverina,864,NaN,NaN,NaN,NaN,NaN,NaN,NaN
site_type,6480,4,retail,2664,NaN,NaN,NaN,NaN,NaN,NaN,NaN
staff_count,"6,480.000",NaN,NaN,NaN,49.689,9.000,25.000,38.500,59.000,226.000,36.564
monthly_volume,"6,480.000",NaN,NaN,NaN,"83,776.168","14,565.000","43,922.000","65,121.000","101,311.500","350,000.000","59,098.730"
criticality_score,"6,480.000",NaN,NaN,NaN,3.194,1.000,3.000,3.000,4.000,5.000,0.907
supplier_dependency_score,"6,480.000",NaN,NaN,NaN,52.723,12.100,41.675,52.550,64.975,100.000,17.394
backup_capacity,"6,480.000",NaN,NaN,NaN,0.483,0.092,0.355,0.497,0.617,0.860,0.180


# Data cleaning and standardisation
The dataset was treated as a standard panel dataset with site-month observations. Cleaning steps focused on validating panel keys, standardising categorical fields, correcting data types, checking range consistency against the synthetic data specification, and handling missing values in a transparent way.

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
)

In [ ]:
df["month"] = pd.to_datetime(df["month"], errors="coerce")

In [ ]:
df["month"].isna().sum()

np.int64(0)

In [ ]:
text_cols = ["site_id", "state", "region", "site_type"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [ ]:
numeric_cols = [
    "staff_count", "monthly_volume", "criticality_score",
    "supplier_dependency_score", "backup_capacity",
    "emergency_plan_maturity", "building_age_years",
    "climate_anomaly_temp_c", "heatwave_days",
    "flood_warning_days", "bushfire_danger_index",
    "storm_outage_hours", "historical_disruptions_3m",
    "historical_disruptions_12m"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
sorted(df["state"].dropna().unique())

['act', 'nsw', 'qld', 'sa', 'tas', 'vic', 'wa']

In [ ]:
sorted(df["site_type"].dropna().unique())

['office', 'retail', 'service centre', 'warehouse']

In [ ]:
df = df.dropna(subset=["month", "site_id"])

In [ ]:
df = df.drop_duplicates(subset=["month", "site_id"])

Data quality check

In [ ]:
#missing critical values
critical_fields = [
    "state", "region", "site_type",
    "staff_count", "monthly_volume", "criticality_score",
    "supplier_dependency_score", "backup_capacity",
    "emergency_plan_maturity", "building_age_years",
    "heatwave_days", "flood_warning_days",
    "bushfire_danger_index", "storm_outage_hours"
]

df[critical_fields].isna().sum().sort_values(ascending=False)


,0
state,0
region,0
site_type,0
staff_count,0
monthly_volume,0
criticality_score,0
supplier_dependency_score,0
backup_capacity,0
emergency_plan_maturity,0
building_age_years,0


In [ ]:
range_rules = {
    "staff_count": (8, 800),
    "monthly_volume": (200, 350000),
    "criticality_score": (1, 5),
    "supplier_dependency_score": (0, 100),
    "backup_capacity": (0, 1),
    "emergency_plan_maturity": (1, 5),
    "building_age_years": (1, 60),
    "climate_anomaly_temp_c": (-2, 4),
    "heatwave_days": (0, 25),
    "flood_warning_days": (0, 12),
    "bushfire_danger_index": (0, 100),
    "storm_outage_hours": (0, 96),
    "historical_disruptions_3m": (0, 3),
    "historical_disruptions_12m": (0, 12),
}
range_issues = {}

for col, (low, high) in range_rules.items():
    invalid = df[(df[col] < low) | (df[col] > high)]
    range_issues[col] = len(invalid)

pd.Series(range_issues).sort_values(ascending=False)

,0
staff_count,0
monthly_volume,0
criticality_score,0
supplier_dependency_score,0
backup_capacity,0
emergency_plan_maturity,0
building_age_years,0
climate_anomaly_temp_c,0
heatwave_days,0
flood_warning_days,0


In [ ]:
(df["historical_disruptions_3m"] > df["historical_disruptions_12m"]).sum()

np.int64(0)

# Feature engineering

In [ ]:
#volume_per_staff
df["volume_per_staff"] = df["monthly_volume"] / df["staff_count"]
df["volume_per_staff"] = np.where(
    df["staff_count"] > 0,
    df["monthly_volume"] / df["staff_count"],
    np.nan
)
#additional flags
df["high_criticality_flag"] = (df["criticality_score"] >= 4).astype(int)
df["high_supplier_dependency_flag"] = (df["supplier_dependency_score"] >= 70).astype(int)
df["low_backup_flag"] = (df["backup_capacity"] < 0.4).astype(int)

## Synthetic target construction

These coefficients are heuristic parameters used to simulate a plausible disruption process, not statistical estimates derived from real-world observations.

In [ ]:
site_type_offset_map = {
    "warehouse": 0.25,
    "retail": 0.10,
    "office": -0.05,
    "service centre": 0.15
}

df["site_type_offset"] = df["site_type"].map(site_type_offset_map).fillna(0)
intercept = -4.8

df["z_disruption"] = (
    intercept
    + 0.055 * df["heatwave_days"]
    + 0.16  * df["flood_warning_days"]
    + 0.012 * df["bushfire_danger_index"]
    + 0.025 * df["storm_outage_hours"]
    + 0.010 * df["supplier_dependency_score"]
    + 0.35  * df["criticality_score"]
    - 1.20  * df["backup_capacity"]
    - 0.30  * df["emergency_plan_maturity"]
    + 0.012 * df["building_age_years"]
    + 0.00018 * df["volume_per_staff"]
    + 0.45  * df["historical_disruptions_3m"]
    + df["site_type_offset"]
)
df["disruption_probability_next_month"] = 1 / (1 + np.exp(-df["z_disruption"]))


In [ ]:
np.random.seed(42)

In [ ]:
df["disruption_flag_next_month"] = np.random.binomial(
    1,
    df["disruption_probability_next_month"]
)

In [ ]:
df["disruption_flag_next_month"].mean()

np.float64(0.07962962962962963)

# Index construction
Three monitoring-oriented composite measures were created. First, a climate risk index summarised site-level climate hazard exposure. Second, a vulnerability score captured operational fragility and resilience weakness. Third, an overall risk score combined exposure, vulnerability, and business impact to support site prioritisation and early-warning analysis.

**Limitation**: These indices are synthetic and assumption-based. They are intended for relative risk ranking and monitoring rather than causal inference or real-world calibration.

In [ ]:
#scale hazard variables
df["heatwave_scaled"] = df["heatwave_days"].clip(0, 25) / 25
df["flood_scaled"] = df["flood_warning_days"].clip(0, 12) / 12
df["bushfire_scaled"] = df["bushfire_danger_index"].clip(0, 100) / 100
df["storm_scaled"] = df["storm_outage_hours"].clip(0, 96) / 96
df["temp_scaled"] = (df["climate_anomaly_temp_c"].clip(-2, 4) + 2) / 6

In [ ]:
# calculate climate_risk_index
df["climate_risk_index"] = (
    0.25 * df["heatwave_scaled"] +
    0.25 * df["flood_scaled"] +
    0.20 * df["bushfire_scaled"] +
    0.20 * df["storm_scaled"] +
    0.10 * df["temp_scaled"]
)
df["climate_risk_index_100"] = 100 * df["climate_risk_index"]

In [ ]:
# calculate vulnerability_score
#scale
df["supplier_dep_scaled"] = df["supplier_dependency_score"].clip(0, 100) / 100
df["building_age_scaled"] = df["building_age_years"].clip(1, 60) / 60
df["hist_3m_scaled"] = df["historical_disruptions_3m"].clip(0, 3) / 3
df["hist_12m_scaled"] = df["historical_disruptions_12m"].clip(0, 12) / 12
df["backup_inverse"] = 1 - df["backup_capacity"].clip(0, 1)
df["emergency_inverse"] = 1 - ((df["emergency_plan_maturity"].clip(1, 5) - 1) / 4)
#composite score
df["vulnerability_score"] = (
    0.25 * df["supplier_dep_scaled"] +
    0.15 * df["building_age_scaled"] +
    0.20 * df["hist_3m_scaled"] +
    0.15 * df["hist_12m_scaled"] +
    0.15 * df["backup_inverse"] +
    0.10 * df["emergency_inverse"]
)
df["vulnerability_score_100"] = 100 * df["vulnerability_score"]

In [ ]:
#calculate business impact score
#scale
df["criticality_scaled"] = (df["criticality_score"].clip(1, 5) - 1) / 4
df["monthly_volume_scaled"] = (df["monthly_volume"].clip(200, 350000) - 200) / (350000 - 200)
df["staff_scaled"] = (df["staff_count"].clip(8, 800) - 8) / (800 - 8)
#composite
df["business_impact_score"] = (
    0.50 * df["criticality_scaled"] +
    0.35 * df["monthly_volume_scaled"] +
    0.15 * df["staff_scaled"]
)
df["business_impact_score_100"] = 100 * df["business_impact_score"]

In [ ]:
#overall risk score
df["overall_risk_score"] = (
    0.40 * df["climate_risk_index"] +
    0.35 * df["vulnerability_score"] +
    0.25 * df["business_impact_score"]
)
df["overall_risk_score_100"] = 100 * df["overall_risk_score"]

In [ ]:
#risk tier
df["risk_tier"] = pd.cut(
    df["overall_risk_score_100"],
    bins=[-np.inf, 33, 66, np.inf],
    labels=["low", "medium", "high"]
)


In [ ]:
#quick validation
score_cols = [
    "climate_risk_index_100",
    "vulnerability_score_100",
    "business_impact_score_100",
    "overall_risk_score_100"
]

df[score_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
climate_risk_index_100,"6,480.000",26.236,5.544,9.275,21.942,26.467,30.685,41.292
vulnerability_score_100,"6,480.000",31.159,7.847,15.405,25.815,30.240,35.435,69.555
business_impact_score_100,"6,480.000",36.583,14.394,3.503,28.802,33.372,44.655,87.102
overall_risk_score_100,"6,480.000",30.546,5.171,15.141,27.036,30.126,33.623,52.222


In [ ]:
#tier distribution
df["risk_tier"].value_counts(dropna=False)

,count
risk_tier,
low,4617
medium,1863
high,0


In [ ]:
df.groupby("site_type")[score_cols].mean().round(2)

,climate_risk_index_100,vulnerability_score_100,business_impact_score_100,overall_risk_score_100
site_type,,,,
office,26.370,24.950,28.440,26.390
retail,26.510,33.120,31.260,30.010
service centre,26.220,32.500,35.470,30.730
warehouse,25.660,31.780,52.940,34.620


In [ ]:
df.groupby("state")[score_cols].mean().round(2).sort_values("overall_risk_score_100", ascending=False)

,climate_risk_index_100,vulnerability_score_100,business_impact_score_100,overall_risk_score_100
state,,,,
wa,27.810,37.480,37.240,33.550
act,27.070,34.000,40.050,32.740
qld,22.540,32.530,40.130,30.430
vic,27.060,29.110,37.470,30.380
nsw,26.960,30.020,36.270,30.360
sa,27.710,32.960,29.510,30.000
tas,22.080,25.800,27.490,24.730


In [ ]:
df.to_csv("/content/drive/MyDrive/ESG Project/Data/climate_business_sites_processed.csv", index=False)